# 08 — FastAPI Endpoints
Test /health, /query, /query/stream, /history, /agents/status using TestClient.

In [ ]:
import sys; sys.path.insert(0, '/home/claude/codebase/code/src')
import os; os.environ['ENABLE_MOCK']='true'; os.environ['REDIS_ENABLED']='false'; os.environ.setdefault('GROQ_API_KEY','')

## Setup TestClient

In [ ]:
from starlette.testclient import TestClient
from api.app import app
client = TestClient(app)
print('FastAPI app ready:', app.title)

## GET /health

In [ ]:
r = client.get('/health')
print('Status:', r.status_code)
print('Body:', r.json())

## POST /query

In [ ]:
r = client.post('/query', json={
    'query': 'What is the current GRR for retention?',
    'thread_id': 'nb-test-01',
    'data_products': ['retention']
})
print('Status:', r.status_code)
data = r.json()
print('Query ID:', data['query_id'])
print('Thread ID:', data['thread_id'])
print('Intent:', data['intent'])
print('Confidence:', data['confidence'])
print('Summary:', data['summary'][:100])

### Validation — empty query returns 400

In [ ]:
r_empty = client.post('/query', json={'query': ''})
print('Empty query status:', r_empty.status_code)  # 400

r_whitespace = client.post('/query', json={'query': '   '})
print('Whitespace query status:', r_whitespace.status_code)  # 400

## GET /agents/status

In [ ]:
r = client.get('/agents/status')
data = r.json()
print('Status:', data['status'])
print('Version:', data['version'])
print('Redis OK:', data['redis_ok'])
print('Agents:')
for a in data['agents']:
    print(f"  {a['name']}: {a['status']}")

## GET /history/{thread_id}

In [ ]:
r = client.get('/history/nb-test-01')
print('Status:', r.status_code)
data = r.json()
print('Thread:', data['thread_id'])
print('Turns:', data['turns'])

## POST /query/stream — SSE events

In [ ]:
import json
events = []
with client.stream('POST', '/query/stream', json={'query': 'What is GRR?'}) as r:
    print('Content-Type:', r.headers['content-type'])
    for line in r.iter_lines():
        if line.startswith('data:'):
            events.append(json.loads(line[5:].strip()))

print(f'Events received: {len(events)}')
for e in events:
    print(f"  type={e.get('type')}")

## Teams bot endpoints

In [ ]:
# Teams health check
r = client.get('/teams/health')
print('Teams health:', r.json())

# Teams message
activity = {
    'type': 'message',
    'text': 'What is the GRR for retention?',
    'from': {'id': 'user-1', 'name': 'Test User'},
    'conversation': {'id': 'conv-nb-01'}
}
r = client.post('/teams/webhook', json=activity)
print('Teams webhook status:', r.status_code)
body = r.json()
print('Response type:', body.get('type'))
print('Has attachments:', len(body.get('attachments', [])) > 0)